# Plot

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(rstatix)
library(emmeans)
library(tibble)
library(tidyr)

In [ ]:
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

## 0. Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/ZVAD_cellnumber/outdir_resegmented"

analysis_summary_files = c(
'/ceph.groups/mshahbazi.grp/rsakata/EXP80/cellpose/plots_resegmented/summarised_results.csv',
'/ceph.groups/mshahbazi.grp/rsakata/EXP79/cellpose/plots_resegmented/summarised_results.csv'
)

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )


In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F62", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP_group = c("pos"= "#86AB30","neg"="#8d8d8dff")

col_GFP_group = c("pos"= "#EB5951","neg"="#8d8d8dff")

col_ZVAD_group = c("pos"= "#005493","neg"="#8d8d8dff")


col_GATA3 = "#489C9C"
col_NANOG = "#EA9542"
col_neg    = "#8d8d8dff"

col_GFP = "#86AB30"
col_GFP = "#EB5951"

## 1. Extract summary files

In [ ]:
# Read and combine all files into one dataframe
merged_df <- analysis_summary_files %>%
  map_dfr(
    ~ read_csv(
      .x,
      col_types = cols(
        ObjectID = col_character(),
        .default = col_guess()
      ),
      show_col_types = FALSE
    )
  )

In [ ]:
head(merged_df)

In [ ]:
tbl <- merged_df %>%
  group_by(sample, sample_name) %>%
  summarise(n_images = n_distinct(image), .groups = "drop") %>%
  arrange(sample)  # optional

tbl

## 2. Preprocess

## Numbers 

In [ ]:
summary_df <- merged_df %>%
  group_by(image, sample_name, condition, Experiment, ZVAD) %>%
  summarise(
    n = n(),
    #n_GATA3 = sum(GATA3_norm > thr$GATA3_norm, na.rm = TRUE),
    #n_NANOG = sum(NANOG_norm > thr$NANOG_norm, na.rm = TRUE),
    n_GFP = mean(GFP == "pos", na.rm = TRUE),
    #pct_mcherry = mean(Mean_mcherry> thr$mcherry_norm, na.rm = TRUE),
    #n_negative = sum(NANOG_norm < thr$NANOG_norm & GATA3_norm < thr$GATA3_norm),
    #pct_double_GFP_caspase3 = 100 * mean(GFP_norm > thr$GFP_norm & caspase3_norm > thr$caspase3_norm),
    .groups = "drop"
  )

head(summary_df)

In [ ]:
title = "counts_per_structure"
w <- 2.5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(summary_df, aes(x = condition, y =n, fill= ZVAD)) +  # dots for each file
    stat_summary( aes(fill = ZVAD), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
       alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color= "grey50"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of cells",
      x = ""
    )+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
      #facet_wrap(~ZVAD) +
      scale_fill_manual(values=col_ZVAD_group)+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
      #legend.position = "none"
    ) 

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
library(dplyr)
library(tidyr)
library(purrr)

check_test <- function(data, group_var = "ZVAD", value_var = "n",
                        conditions = c("neg", "pos"), alpha = 0.05) {

  d <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    droplevels()

  d %>%
    group_by( condition) %>%
    group_modify(~ {
      g <- split(.x[[value_var]], .x[[group_var]])
      g <- g[conditions]                       # keep the two groups in order

      # need at least 3 non-NA points per group for Shapiro
      n_ok <- all(sapply(g, function(x) sum(!is.na(x)) >= 3))

      if (!n_ok) {
        return(tibble(
          n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
          shapiro_p1 = NA_real_, shapiro_p2 = NA_real_,
          levene_p = NA_real_, normal = NA,
          recommended = "too few points (use Wilcoxon / be cautious)"
        ))
      }

      # normality per group
      sp1 <- shapiro.test(g[[1]])$p.value
      sp2 <- shapiro.test(g[[2]])$p.value
      normal <- (sp1 > alpha) & (sp2 > alpha)

      # equal-variance check (F-test; swap for car::leveneTest if preferred)
      var_p <- tryCatch(var.test(g[[1]], g[[2]])$p.value, error = function(e) NA_real_)

      rec <- if (normal) {
        if (!is.na(var_p) && var_p > alpha) "Student t-test (var.equal = TRUE)"
        else "Welch t-test"
      } else {
        "Wilcoxon rank-sum test"
      }

      tibble(
        n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
        shapiro_p1 = sp1, shapiro_p2 = sp2,
        levene_p = var_p, normal = normal,
        recommended = rec
      )
    }) %>%
    ungroup()
}

# usage
check_test(summary_df)

In [ ]:
wilcox_res <- summary_df %>%
  #filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")) %>%
  #mutate(sample_name = droplevels(sample_name)) %>%
  group_by(condition) %>%
  wilcox_test(
    n ~ ZVAD,
    #ref.group = "G_R",
    p.adjust.method = "BH"
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
summary_df <- merged_df %>%
  group_by(image, sample_name, condition, Experiment, ZVAD,  GFP) %>%
  summarise(
    n = n(),
    #n_GATA3 = sum(GATA3_norm > thr$GATA3_norm, na.rm = TRUE),
    #n_NANOG = sum(NANOG_norm > thr$NANOG_norm, na.rm = TRUE),
    #n_GFP = mean(GFP == "pos", na.rm = TRUE),
    #pct_mcherry = mean(Mean_mcherry> thr$mcherry_norm, na.rm = TRUE),
    #n_negative = sum(NANOG_norm < thr$NANOG_norm & GATA3_norm < thr$GATA3_norm),
    #pct_double_GFP_caspase3 = 100 * mean(GFP_norm > thr$GFP_norm & caspase3_norm > thr$caspase3_norm),
    .groups = "drop"
  )

head(summary_df)

In [ ]:
title = "counts_per_structure_GFPgroup"
w <- 2.5
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
  
  summary_df_sub = summary_df %>% filter(GFP == "neg")

p = ggplot(  summary_df_sub, aes(x = condition, y =n, fill= ZVAD)) +  # dots for each file
    stat_summary( aes(fill = ZVAD), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
       alpha = 0.4, width = 0.6) +   # error bars
    geom_jitter(
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color= "grey50"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "number of cells",
      x = ""
    )+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
      #facet_wrap(~ZVAD) +
      scale_fill_manual(values=col_ZVAD_group)+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
      #legend.position = "none"
    ) +facet_wrap(~GFP)

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
wilcox_res <- summary_df_sub %>%
  #filter(sample_name %in% c("D6_RrevG_D", "D6_RrevG_F")) %>%
  #mutate(sample_name = droplevels(sample_name)) %>%
  group_by(condition) %>%
  wilcox_test(
    n ~ ZVAD,
    #ref.group = "G_R",
    p.adjust.method = "BH"
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
head(merged_df)

In [ ]:
merged_df$EXP = EXP
write_csv(merged_df, file.path(out_dir, "summarised_results.csv"))